# ECGR 4105 Homework 6: Fully Connected Neural Networks

**Name:** Enrique Guevara Mendoza  
**Student ID:** 801144054  
**GitHub repository:** https://github.com/egmendoza/ecgr-4105-hw6

Problem 1 trains a fully connected neural network to predict housing price and compares it with linear regression and SVR.
Problem 2 trains a fully connected neural network to classify breast tumours and compares it with logistic regression and SVM.
The networks are built with `nn.Sequential` and trained with the training/validation loop from the lectures.

In [ ]:
import json, os, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, SVC
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
torch.manual_seed(SEED)
os.makedirs("figures", exist_ok=True)
results = {}

def to_t(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32)

def build_network(n_inputs, hidden, activation, sigmoid_output=False):
    """n_inputs -> hidden[0] -> activation -> hidden[1] -> activation -> 1 output"""
    torch.manual_seed(SEED)                     # same starting weights every time this is called
    layers, prev = [], n_inputs
    for n in hidden:
        layers += [nn.Linear(prev, n), activation()]
        prev = n
    layers.append(nn.Linear(prev, 1))
    if sigmoid_output:
        layers.append(nn.Sigmoid())             # output becomes a probability between 0 and 1
    return nn.Sequential(*layers)

def training_loop(n_epochs, optimizer, model, loss_fn, X_train, y_train, X_val, y_val, classification=False):
    """Training loop from Lecture 9, using the whole training set in every step.
    It also saves a copy of the weights from the epoch with the lowest validation loss."""
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss, best_weights, best_epoch = float("inf"), None, 0
    for epoch in range(1, n_epochs + 1):
        train_pred = model(X_train)
        train_loss = loss_fn(train_pred, y_train)

        with torch.no_grad():                   # validation: no gradients needed
            val_pred = model(X_val)
            val_loss = loss_fn(val_pred, y_val)

        optimizer.zero_grad()
        train_loss.backward()                   # gradients come from the training loss only
        optimizer.step()

        hist["train_loss"].append(train_loss.item())
        hist["val_loss"].append(val_loss.item())
        if classification:
            hist["train_acc"].append(((train_pred >= 0.5).float() == y_train).float().mean().item())
            hist["val_acc"].append(((val_pred >= 0.5).float() == y_val).float().mean().item())
        if val_loss.item() < best_val_loss:
            best_val_loss, best_epoch = val_loss.item(), epoch
            best_weights = copy.deepcopy(model.state_dict())
        if epoch == 1 or epoch % 100 == 0:
            print(f"Epoch {epoch:4d}, Training loss {train_loss.item():.4f}, Validation loss {val_loss.item():.4f}")
    model.load_state_dict(best_weights)          # go back to the best epoch
    return hist, best_epoch, best_val_loss

## Problem 1: Housing price prediction

### Preprocessing
* The six yes/no columns are mapped to 1/0. `furnishingstatus` has three categories, so it is turned into two yes/no columns,
  `furnished` and `semi-furnished` (a house with both 0 is unfurnished). This gives **13 inputs**.
* **Train/test split:** 80% / 20% with `random_state=42` (436 / 109 houses), the same split as Homework 4. All models use it,
  and the test set is only used at the very end.
* **Validation set:** 20% of the training set (88 houses) is held out, as in Lectures 3 and 9. It is used to compare the network
  settings, to pick the training epoch, and to pick the SVR kernel. The models are fit on the other 348 houses.
* **Standardization:** inputs are standardized with the mean and standard deviation of the training set. The price is standardized
  too for the network and SVR, and predictions are converted back to price (in millions) before computing MSE, RMSE and R².

In [ ]:
housing = pd.read_csv("data/Housing.csv")
binary = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]
housing[binary] = housing[binary].apply(lambda c: c.map({"yes": 1, "no": 0}))
housing["furnished"] = (housing["furnishingstatus"] == "furnished").astype(int)
housing["semi-furnished"] = (housing["furnishingstatus"] == "semi-furnished").astype(int)
H_FEATURES = ["area", "bedrooms", "bathrooms", "stories", "mainroad", "guestroom", "basement", "hotwaterheating",
              "airconditioning", "parking", "prefarea", "furnished", "semi-furnished"]
Xh = housing[H_FEATURES].values.astype(float)
yh = housing["price"].values / 1e6                       # price in millions

Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=SEED)
x_scaler = StandardScaler().fit(Xh_train)
y_scaler = StandardScaler().fit(yh_train.reshape(-1, 1))
Xh_train_s, Xh_test_s = x_scaler.transform(Xh_train), x_scaler.transform(Xh_test)
yh_train_s = y_scaler.transform(yh_train.reshape(-1, 1)).ravel()

Xh_fit, Xh_val, yh_fit, yh_val = train_test_split(Xh_train_s, yh_train_s, test_size=0.2, random_state=SEED)
print(f"Training {len(Xh_train)} houses (fit {len(Xh_fit)} + validation {len(Xh_val)}), test {len(Xh_test)}")

def to_price(p):
    return y_scaler.inverse_transform(np.asarray(p).reshape(-1, 1)).ravel()

def reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MSE": mse, "RMSE": float(np.sqrt(mse)), "R2": r2_score(y_true, y_pred)}

Xf, yf, Xv, yv = to_t(Xh_fit), to_t(yh_fit).unsqueeze(1), to_t(Xh_val), to_t(yh_val).unsqueeze(1)

FileNotFoundError: [Errno 2] No such file or directory: 'data/Housing.csv'

### 1.a Neural network experiments
All networks have two hidden layers and are trained with the Adam optimizer (Lecture 8) and MSE loss for 1000 epochs, using the whole
fit set in every step. I started from a baseline of 32 and 16 neurons with ReLU and a learning rate of 0.001, then changed one
setting at a time. The settings are compared on the **validation** loss (standardized price units).

In [ ]:
EPOCHS = 1000
EXPERIMENTS = [
    ("Baseline",            (32, 16), nn.ReLU, 1e-3),
    ("More neurons",        (64, 32), nn.ReLU, 1e-3),
    ("Fewer neurons",       (16, 8),  nn.ReLU, 1e-3),
    ("Higher learning rate", (32, 16), nn.ReLU, 1e-2),
    ("Tanh activation",     (32, 16), nn.Tanh, 1e-3),
]
h_rows = []
for name, hidden, act, lr in EXPERIMENTS:
    print(f"\n===== {name}: hidden {hidden}, {act.__name__}, lr = {lr} =====")
    model = build_network(len(H_FEATURES), hidden, act)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    _, best_epoch, best_val = training_loop(EPOCHS, optimizer, model, nn.MSELoss(), Xf, yf, Xv, yv)
    h_rows.append({"experiment": name, "hidden": str(hidden), "activation": act.__name__, "lr": lr,
                   "best_epoch": best_epoch, "val_loss": best_val})
h_exp = pd.DataFrame(h_rows)
print("\n", h_exp.round(4).to_string(index=False))

In [ ]:
best = h_exp.loc[h_exp["val_loss"].idxmin()]
H_HIDDEN, H_ACT, H_LR = eval(best["hidden"]), {"ReLU": nn.ReLU, "Tanh": nn.Tanh}[best["activation"]], float(best["lr"])
print(f"Selected: {best['experiment']} (hidden {H_HIDDEN}, {best['activation']}, lr = {H_LR})")

h_model = build_network(len(H_FEATURES), H_HIDDEN, H_ACT)
print(h_model)
h_n_params = sum(p.numel() for p in h_model.parameters())
print("Learnable parameters:", h_n_params)
h_hist, h_best_epoch, _ = training_loop(EPOCHS, optim.Adam(h_model.parameters(), lr=H_LR), h_model, nn.MSELoss(), Xf, yf, Xv, yv)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, EPOCHS + 1), h_hist["train_loss"], label="Training loss")
ax.plot(range(1, EPOCHS + 1), h_hist["val_loss"], label="Validation loss")
ax.axvline(h_best_epoch, color="gray", ls="--", lw=1, label=f"lowest validation loss (epoch {h_best_epoch})")
ax.set(xlabel="Epoch", ylabel="MSE (standardized price)", title="Problem 1: Neural network training and validation loss")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig("figures/p1_nn_loss.png", dpi=150); plt.show()

with torch.no_grad():
    nn_pred = to_price(h_model(to_t(Xh_test_s)).numpy())
    nn_train_r2 = r2_score(yh_train, to_price(h_model(to_t(Xh_train_s)).numpy()))
nn_h = reg_metrics(yh_test, nn_pred)
print(f"Weights from epoch {h_best_epoch}. Training-set R2 {nn_train_r2:.4f}")
print("Neural network test:", {k: round(v, 4) for k, v in nn_h.items()})

### 1.b Comparison with linear regression and SVR
* **Linear regression:** the Homework 2 method, gradient descent with a hand-derived gradient in NumPy (learning rate 0.1, 5000
  iterations, parameters start at zero), trained on the full training set.
* **SVR:** the kernel is chosen on the same validation set as the network (C = 1, the default), then the chosen SVR is retrained on
  the full training set.

In [ ]:
def linreg_gd(X, y, lr=0.1, iters=5000):
    Xb = np.c_[np.ones(len(X)), X]
    theta = np.zeros(Xb.shape[1])
    for _ in range(iters):
        theta -= lr * (2 / len(y)) * Xb.T @ (Xb @ theta - y)
    return theta

theta = linreg_gd(Xh_train_s, yh_train)
lin_pred = np.c_[np.ones(len(Xh_test_s)), Xh_test_s] @ theta
lin_h = reg_metrics(yh_test, lin_pred)
print("Linear regression test:", {k: round(v, 4) for k, v in lin_h.items()})

svr_val = {}
for kernel in ["linear", "rbf", "poly"]:
    m = SVR(kernel=kernel).fit(Xh_fit, yh_fit)
    svr_val[kernel] = mean_squared_error(yh_val, m.predict(Xh_val))
print("SVR validation MSE by kernel:", {k: round(v, 4) for k, v in svr_val.items()})
svr_kernel = min(svr_val, key=svr_val.get)
svr = SVR(kernel=svr_kernel).fit(Xh_train_s, yh_train_s)
svr_pred = to_price(svr.predict(Xh_test_s))
svr_h = reg_metrics(yh_test, svr_pred)
print(f"SVR ({svr_kernel}) test:", {k: round(v, 4) for k, v in svr_h.items()})

h_comp = pd.DataFrame({"Linear regression": lin_h, f"SVR ({svr_kernel})": svr_h, "Neural network": nn_h}).T
print("\nTest-set comparison (MSE in millions^2, RMSE in millions):\n", h_comp.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)
lim = [1, 11]
for a, (name, p) in zip(axes, [("Linear regression", lin_pred), ("SVR", svr_pred), ("Neural network", nn_pred)]):
    a.scatter(yh_test, p, s=16, alpha=0.7)
    a.plot(lim, lim, "r--", lw=1, label="ideal")
    a.set(title=f"{name}, R$^2$ = {r2_score(yh_test, p):.3f}", xlabel="Actual price (millions)")
    a.grid(alpha=0.3)
axes[0].set_ylabel("Predicted price (millions)"); axes[0].legend()
fig.suptitle("Problem 1: Predicted vs. actual price on the test set")
fig.tight_layout(); fig.savefig("figures/p1_pred_vs_actual.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for a, m, t in zip(axes, ["MSE", "RMSE", "R2"], ["Test MSE (millions$^2$)", "Test RMSE (millions)", "Test R$^2$"]):
    h_comp[m].plot(kind="bar", ax=a, color=["C0", "C1", "C2"])
    a.set_title(t); a.set_xticklabels(["Linear", "SVR", "Neural net"], rotation=0); a.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig("figures/p1_comparison.png", dpi=150); plt.show()

results["p1"] = {"experiments": h_exp.to_dict(orient="records"), "selected": best["experiment"],
                 "hidden": list(H_HIDDEN), "activation": best["activation"], "lr": H_LR, "epochs": EPOCHS,
                 "best_epoch": h_best_epoch, "n_params": h_n_params, "nn": nn_h, "nn_train_r2": nn_train_r2,
                 "linear": lin_h, "svr": svr_h, "svr_kernel": svr_kernel, "svr_val": svr_val}

## Problem 2: Breast cancer classification

### Preprocessing
* The `id` column and the empty trailing column are dropped, leaving the **30 features**. Malignant = 1, benign = 0.
* **Train/test split:** 80% / 20%, stratified, `random_state=42` (455 / 114 tumours), the same split as Homeworks 3 and 4.
* **Validation set:** 20% of the training set (91 tumours), used the same way as in Problem 1. The models are fit on the other 364.
* **Standardization** with the training-set mean and standard deviation.

In [ ]:
cancer = pd.read_csv("data/Cancer.csv")
cancer = cancer.drop(columns=[c for c in cancer.columns if c == "id" or c.startswith("Unnamed")])
yc = (cancer["diagnosis"] == "M").astype(int).values
Xc = cancer.drop(columns="diagnosis").values
assert Xc.shape[1] == 30

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=SEED, stratify=yc)
c_scaler = StandardScaler().fit(Xc_train)
Xc_train_s, Xc_test_s = c_scaler.transform(Xc_train), c_scaler.transform(Xc_test)
Xc_fit, Xc_val, yc_fit, yc_val = train_test_split(Xc_train_s, yc_train, test_size=0.2, random_state=SEED, stratify=yc_train)
print(f"Training {len(Xc_train)} tumours (fit {len(Xc_fit)} + validation {len(Xc_val)}), "
      f"test {len(Xc_test)} ({yc_test.sum()} malignant, {len(yc_test) - yc_test.sum()} benign)")

def clf_metrics(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred), "F1": f1_score(y_true, y_pred)}

Cf, cf, Cv, cv = to_t(Xc_fit), to_t(yc_fit).unsqueeze(1), to_t(Xc_val), to_t(yc_val).unsqueeze(1)

### 2.a Neural network experiments
The network ends with a sigmoid, so its output is the probability that a tumour is malignant, and it is trained with binary
cross-entropy (`nn.BCELoss`, the logistic regression loss from Lecture 4). A tumour is classified as malignant when the probability
is 0.5 or higher. The same five experiments as Problem 1 are compared on validation loss.

In [ ]:
c_rows = []
for name, hidden, act, lr in EXPERIMENTS:
    print(f"\n===== {name}: hidden {hidden}, {act.__name__}, lr = {lr} =====")
    model = build_network(30, hidden, act, sigmoid_output=True)
    hist, best_epoch, best_val = training_loop(EPOCHS, optim.Adam(model.parameters(), lr=lr), model, nn.BCELoss(),
                                               Cf, cf, Cv, cv, classification=True)
    c_rows.append({"experiment": name, "hidden": str(hidden), "activation": act.__name__, "lr": lr,
                   "best_epoch": best_epoch, "val_loss": best_val, "val_acc_at_best": hist["val_acc"][best_epoch - 1]})
c_exp = pd.DataFrame(c_rows)
print("\n", c_exp.round(4).to_string(index=False))

In [ ]:
best = c_exp.loc[c_exp["val_loss"].idxmin()]
C_HIDDEN, C_ACT, C_LR = eval(best["hidden"]), {"ReLU": nn.ReLU, "Tanh": nn.Tanh}[best["activation"]], float(best["lr"])
print(f"Selected: {best['experiment']} (hidden {C_HIDDEN}, {best['activation']}, lr = {C_LR})")
c_model = build_network(30, C_HIDDEN, C_ACT, sigmoid_output=True)
print(c_model)
c_n_params = sum(p.numel() for p in c_model.parameters())
print("Learnable parameters:", c_n_params)
c_hist, c_best_epoch, _ = training_loop(EPOCHS, optim.Adam(c_model.parameters(), lr=C_LR), c_model, nn.BCELoss(),
                                        Cf, cf, Cv, cv, classification=True)

ep = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].plot(ep, c_hist["train_loss"], label="Training"); axes[0].plot(ep, c_hist["val_loss"], label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Binary cross-entropy", title="Loss")
axes[1].plot(ep, c_hist["train_acc"], label="Training"); axes[1].plot(ep, c_hist["val_acc"], label="Validation")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Classification accuracy", ylim=(0.85, 1.005))
for a in axes:
    a.axvline(c_best_epoch, color="gray", ls="--", lw=1, label=f"lowest validation loss (epoch {c_best_epoch})")
    a.grid(alpha=0.3); a.legend()
fig.suptitle("Problem 2: Neural network training curves")
fig.tight_layout(); fig.savefig("figures/p2_nn_curves.png", dpi=150); plt.show()

with torch.no_grad():
    nn_c_pred = (c_model(to_t(Xc_test_s)).numpy().ravel() >= 0.5).astype(int)
    nn_c_train_acc = accuracy_score(yc_train, (c_model(to_t(Xc_train_s)).numpy().ravel() >= 0.5).astype(int))
nn_c = clf_metrics(yc_test, nn_c_pred)
print(f"Weights from epoch {c_best_epoch}. Training-set accuracy {nn_c_train_acc:.4f}")
print("Neural network test:", {k: round(v, 4) for k, v in nn_c.items()})

### 2.b Comparison with logistic regression and SVM
* **Logistic regression:** my Homework 3 implementation (gradient descent with an L2 penalty, λ = 3 as chosen in Homework 3,
  learning rate 0.1, 20,000 iterations), trained on the full training set.
* **SVM:** the kernel is chosen on the validation set (C = 1, the default), then retrained on the full training set.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def train_logreg(X, y, lr=0.1, iters=20000, lam=0.0):
    m, n = X.shape
    theta, b = np.zeros(n), 0.0
    for _ in range(iters):
        err = sigmoid(X @ theta + b) - y
        theta -= lr * ((X.T @ err) / m + (lam / m) * theta)
        b -= lr * err.mean()
    return theta, b

theta_c, b_c = train_logreg(Xc_train_s, yc_train, lam=3)
lr_c_pred = (sigmoid(Xc_test_s @ theta_c + b_c) >= 0.5).astype(int)
lr_c = clf_metrics(yc_test, lr_c_pred)
print("Logistic regression test:", {k: round(v, 4) for k, v in lr_c.items()})

svm_val = {}
for kernel in ["linear", "rbf", "poly", "sigmoid"]:
    svm_val[kernel] = accuracy_score(yc_val, SVC(kernel=kernel).fit(Xc_fit, yc_fit).predict(Xc_val))
print("SVM validation accuracy by kernel:", {k: round(v, 4) for k, v in svm_val.items()})
svm_kernel = max(svm_val, key=svm_val.get)              # ties go to the first (simplest) kernel
svm_c_pred = SVC(kernel=svm_kernel).fit(Xc_train_s, yc_train).predict(Xc_test_s)
svm_c = clf_metrics(yc_test, svm_c_pred)
print(f"SVM ({svm_kernel}) test:", {k: round(v, 4) for k, v in svm_c.items()})

c_comp = pd.DataFrame({"Logistic regression": lr_c, f"SVM ({svm_kernel})": svm_c, "Neural network": nn_c}).T
print("\nTest-set comparison:\n", c_comp.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
cms = {}
for a, (name, p) in zip(axes, [("Logistic regression", lr_c_pred), ("SVM", svm_c_pred), ("Neural network", nn_c_pred)]):
    cm = confusion_matrix(yc_test, p); cms[name] = cm.tolist()
    ConfusionMatrixDisplay(cm, display_labels=["Benign (0)", "Malignant (1)"]).plot(ax=a, cmap="Blues", colorbar=False)
    a.set_title(name)
fig.suptitle("Problem 2: Confusion matrices on the test set")
fig.tight_layout(); fig.savefig("figures/p2_confusion.png", dpi=150); plt.show()

ax = c_comp.plot(kind="bar", figsize=(9, 4.3), width=0.8)
ax.set(ylim=(0.85, 1.005), ylabel="Test score", title="Problem 2: Test classification performance")
ax.set_xticklabels(["Logistic regression", "SVM", "Neural network"], rotation=0); ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right"); plt.tight_layout(); plt.savefig("figures/p2_comparison.png", dpi=150); plt.show()

results["p2"] = {"experiments": c_exp.to_dict(orient="records"), "selected": best["experiment"],
                 "hidden": list(C_HIDDEN), "activation": best["activation"], "lr": C_LR, "epochs": EPOCHS,
                 "best_epoch": c_best_epoch, "n_params": c_n_params, "nn": nn_c, "nn_train_acc": nn_c_train_acc,
                 "logreg": lr_c, "svm": svm_c, "svm_kernel": svm_kernel, "svm_val": svm_val, "confusion": cms}
with open("results.json", "w") as f:
    json.dump(results, f, indent=1, default=float)
print("Saved results.json")